# 1. Import Thư Viện & Setup Cấu Hình
## Cấu hình tự động nhận diện Project Root.

In [ ]:
import os
import sys
import time
from pathlib import Path
import pandas as pd
import numpy as np

# Tự động tìm thư mục 'backend'
current_dir = Path.cwd().resolve()
while current_dir.name != "backend" and current_dir.parent != current_dir:
    if (current_dir / "backend").exists():
        current_dir = current_dir / "backend"
        break
    current_dir = current_dir.parent

if str(current_dir) not in sys.path:
    sys.path.insert(0, str(current_dir))

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV

from app.ml.preprocessing.breast_cancer import load_breast_cancer_dataset

PROCESSED_DATA_PATH = current_dir.parent / "data/processed/breast_cancer_cleaned.csv"
RAW_DATA_PATH = current_dir.parent / "data/raw/uci_wdbc/wdbc.data"


# 2. So sánh 3 bộ Dataset với Sklearn Baseline (Unpruned Tree)
## Đánh giá Accuracy, Overfit Gap, F1... cho 3 dataset.

In [2]:
datasets = {
    "Processed Data": PROCESSED_DATA_PATH,
    "Raw Data (UCI)": RAW_DATA_PATH,
    "Sklearn Data": None
}

results = []

for name, path in datasets.items():
    X = y = None
    if name == "Sklearn Data":
        raw = load_breast_cancer()
        X = raw.data
        y = np.where(raw.target == 0, 1, 0)
    else:
        if name == "Raw Data (UCI)" and path.exists():
            ds = load_breast_cancer_dataset(path)
            X = ds.features.to_numpy(dtype=float)
            y = (ds.target == "M").astype(int).to_numpy()
        elif path.exists():
            frame = pd.read_csv(path)
            feature_cols = [c for c in frame.columns if c != "diagnosis"]
            X = frame[feature_cols].to_numpy(dtype=float)
            y = frame["diagnosis"].to_numpy(dtype=int)
    
    if X is None or y is None:
        print(f"Skipping {name}, data not found.")
        continue
            
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    model = DecisionTreeClassifier(random_state=42)
    start_time = time.time()
    model.fit(X_train, y_train)
    fit_time = time.time() - start_time
    
    y_train_pred = model.predict(X_train)
    train_acc = accuracy_score(y_train, y_train_pred)
    
    y_test_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)
    test_recall = recall_score(y_test, y_test_pred)
    test_prec = precision_score(y_test, y_test_pred)
    
    def calc_gini(y_true):
        _, counts = np.unique(y_true, return_counts=True)
        probs = counts / len(y_true)
        return 1.0 - np.sum(probs ** 2)
    
    train_gini = calc_gini(y_train)
    acc_gap = train_acc - test_acc
    
    results.append({
        "Dataset": name,
        "Accuracy": round(test_acc, 4),
        "Precision": round(test_prec, 4),
        "Recall": round(test_recall, 4),
        "F1 Score": round(test_f1, 4),
        "Fit Time (s)": round(fit_time, 5),
        "Train Acc": round(train_acc, 4),
        "Overfit Gap (Acc)": round(acc_gap, 4),
        "Root Gini": round(train_gini, 4)
    })

results_df = pd.DataFrame(results)
display(results_df)

,Dataset,Accuracy,Precision,Recall,F1 Score,Fit Time (s),Train Acc,Overfit Gap (Acc),Root Gini
0,Processed Data,0.9035,0.9189,0.8095,0.8608,0.00870,1.0,0.0965,0.4681
1,Raw Data (UCI),0.9298,0.9048,0.9048,0.9048,0.00527,1.0,0.0702,0.4681
2,Sklearn Data,0.9298,0.9048,0.9048,0.9048,0.00557,1.0,0.0702,0.4681


# 3. Grid Search & 5-Fold Cross Validation
## 3.1 Tìm Hyperparameter cho Processed Data.

In [ ]:
print("--- TIẾN HÀNH GRID SEARCH & 5-FOLD CROSS VALIDATION ---")

frame = pd.read_csv(PROCESSED_DATA_PATH)
feature_cols = [c for c in frame.columns if c != "diagnosis"]
X_grid = frame[feature_cols].to_numpy(dtype=float)
y_grid = frame["diagnosis"].to_numpy(dtype=int)

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': list(range(2, 9)),
    'min_samples_split': list(range(2, 20)),
    'min_samples_leaf': list(range(1, 15))
}

dt = DecisionTreeClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=dt, 
    param_grid=param_grid, 
    cv=5, 
    scoring=['accuracy', 'precision', 'f1', 'recall', 'roc_auc'], 
    refit='recall',
    return_train_score=True,
    n_jobs=-1,
    verbose=1
)

start_time = time.time()
grid_search.fit(X_grid, y_grid)
search_time = time.time() - start_time

print(f"Tổng thời gian Grid Search: {search_time:.2f} giây")
print(f"Bộ tham số tối ưu nhất (Best Params): {grid_search.best_params_}")
print(f"Best CV recall Score: {grid_search.best_score_:.4f}\n")

cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results['overfit_gap_acc'] = cv_results['mean_train_accuracy'] - cv_results['mean_test_accuracy']

cols_to_show = [
    'params', 
    'mean_test_accuracy', 
    'mean_test_precision', 
    'mean_test_f1', 
    'mean_test_recall',
    'mean_fit_time', 
    'overfit_gap_acc'
]

top_models = cv_results.sort_values(by='rank_test_f1', ascending=False).head(10)
print("Top 10 Hyperparameters tốt nhất:")
display(top_models[cols_to_show])

--- TIẾN HÀNH GRID SEARCH & 5-FOLD CROSS VALIDATION ---
Fitting 5 folds for each of 3528 candidates, totalling 17640 fits
Tổng thời gian Grid Search: 1335.39 giây
Bộ tham số tối ưu nhất (Best Params): {'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 5, 'min_samples_split': 2}
Best CV F1 Score: 0.9296

Top 10 Hyperparameters tốt nhất:


,params,mean_test_accuracy,mean_test_precision,mean_test_f1,mean_test_recall,mean_fit_time,overfit_gap_acc
3004,"{'criterion': 'entropy', 'max_depth': 6, 'min_...",0.903307,0.895422,0.865547,0.850055,0.005903,0.057148
3005,"{'criterion': 'entropy', 'max_depth': 6, 'min_...",0.903307,0.895422,0.865547,0.850055,0.006294,0.057148
2996,"{'criterion': 'entropy', 'max_depth': 6, 'min_...",0.903307,0.895422,0.865547,0.850055,0.006187,0.057148
2997,"{'criterion': 'entropy', 'max_depth': 6, 'min_...",0.903307,0.895422,0.865547,0.850055,0.006029,0.057148
2998,"{'criterion': 'entropy', 'max_depth': 6, 'min_...",0.903307,0.895422,0.865547,0.850055,0.006994,0.057148
2999,"{'criterion': 'entropy', 'max_depth': 6, 'min_...",0.903307,0.895422,0.865547,0.850055,0.005970,0.057148
3000,"{'criterion': 'entropy', 'max_depth': 6, 'min_...",0.903307,0.895422,0.865547,0.850055,0.006720,0.057148
3001,"{'criterion': 'entropy', 'max_depth': 6, 'min_...",0.903307,0.895422,0.865547,0.850055,0.046321,0.057148
3002,"{'criterion': 'entropy', 'max_depth': 6, 'min_...",0.903307,0.895422,0.865547,0.850055,0.006659,0.057148
3003,"{'criterion': 'entropy', 'max_depth': 6, 'min_...",0.903307,0.895422,0.865547,0.850055,0.006112,0.057148


## 3.2 Tìm Hyperparameter cho raw Data.

In [34]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ==========================================
# 1. LOAD DATA & TRAIN-TEST SPLIT
# ==========================================
# Load bộ dữ liệu chuẩn (tương đương bản UCI)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target  # 0: Malignant (Ác tính), 1: Benign (Lành tính)

# Đổi nhãn y học chuẩn: Thường gán 1 cho lớp nguy hiểm (Malignant) để tính Recall chuẩn xác
# Mặc định của sklearn: 0 = Malignant, 1 = Benign. Ta đảo ngược lại để 1 = Ác tính (Positive class)
y = np.where(y == 0, 1, 0)

# Chia Train-Test bắt buộc phải dùng stratify=y để giữ nguyên tỷ lệ nhãn ở cả 2 tập
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ==========================================
# 2. XÂY DỰNG PIPELINE (CHỐNG DATA LEAKAGE)
# ==========================================
# Quy trình chuẩn: Dữ liệu đi qua bộ Scale trước rồi mới vào Model Decision Tree
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('dt', DecisionTreeClassifier(random_state=42))
])

# ==========================================
# 3. THIẾT LẬP PARAMETER GRID & GRID SEARCH
# ==========================================
# Lưu ý: Khi dùng Pipeline, tên tham số trong param_grid phải có tiền tố "tên_bước__"
param_grid = {
    'dt__criterion': ['gini', 'entropy'],
    'dt__max_depth': [2, 3, 4, 5, 6, 7, 8, 9],
    'dt__min_samples_split': list(range(2, 15)),
    'dt__min_samples_leaf': list(range(1, 10)),
    'dt__class_weight': ['balanced', None] # class_weight rất quan trọng để xử lý dữ liệu lệch
}

# Khởi tạo GridSearch đánh giá đa thang đo, lấy RECALL làm gốc để Refit mô hình
grid_search = GridSearchCV(
    estimator=pipeline, 
    param_grid=param_grid, 
    cv=5, 
    scoring=['accuracy', 'precision', 'f1', 'recall', 'roc_auc'], 
    refit='recall',  # Ép mô hình chọn bộ tham số bắt sót ít bệnh nhân nhất
    return_train_score=True,
    n_jobs=-1,
    verbose=0
)

# Tiến hành Fit dữ liệu Train vào Pipeline
grid_search.fit(X_train, y_train)

# ==========================================
# 4. TRÍCH XUẤT KẾT QUẢ CV CHUẨN XÁC
# ==========================================
best_idx = grid_search.best_index_
print("===== KẾT QUẢ TRÊN TẬP VALIDATION (CROSS-VALIDATION) =====")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Recall (Chính là best_score_): {grid_search.best_score_:.4f}")
print(f"CV Accuracy : {grid_search.cv_results_['mean_test_accuracy'][best_idx]:.4f}")
print(f"CV Precision: {grid_search.cv_results_['mean_test_precision'][best_idx]:.4f}")
print(f"CV F1-Score : {grid_search.cv_results_['mean_test_f1'][best_idx]:.4f}")
print(f"CV ROC AUC  : {grid_search.cv_results_['mean_test_roc_auc'][best_idx]:.4f}\n")

# ==========================================
# 5. ĐÁNH GIÁ THỰC TẾ TRÊN TẬP TEST ĐỘC LẬP
# ==========================================
# Dự đoán kết quả trên tập Test chưa từng thấy bằng mô hình tối ưu nhất
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

print("===== KẾT QUẢ ĐÁNH GIÁ THỰC TẾ TRÊN TẬP TEST =====")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Benign (Lành tính)', 'Malignant (Ác tính)']))
print(f"Test ROC AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")


===== KẾT QUẢ TRÊN TẬP VALIDATION (CROSS-VALIDATION) =====
Best Parameters: {'dt__class_weight': None, 'dt__criterion': 'entropy', 'dt__max_depth': 6, 'dt__min_samples_leaf': 1, 'dt__min_samples_split': 5}
Best CV Recall (Chính là best_score_): 0.9353
CV Accuracy : 0.9451
CV Precision: 0.9199
CV F1-Score : 0.9274
CV ROC AUC  : 0.9450

===== KẾT QUẢ ĐÁNH GIÁ THỰC TẾ TRÊN TẬP TEST =====
Confusion Matrix:
[[71  1]
 [ 7 35]]

Classification Report:
                     precision    recall  f1-score   support

 Benign (Lành tính)       0.91      0.99      0.95        72
Malignant (Ác tính)       0.97      0.83      0.90        42

           accuracy                           0.93       114
          macro avg       0.94      0.91      0.92       114
       weighted avg       0.93      0.93      0.93       114

Test ROC AUC Score: 0.9433


# 4.1 Đánh giá thời gian và Overfitting của Module Tuning (Processed Data)\nKiểm tra thời gian chạy, thời gian suy diễn và độ quá khớp.

In [42]:
import time
from sklearn.model_selection import train_test_split
from app.ml.sklearn_tree.max_depth import run_max_depth_experiment, MaxDepthExperimentConfig
from app.ml.sklearn_tree.gini_vs_entropy import run_gini_vs_entropy_experiment
from app.ml.sklearn_tree.min_samples import run_min_samples_tuning, MinSamplesConfig
from app.ml.sklearn_tree.baseline import BaselineConfig
import pandas as pd

print("--- ĐÁNH GIÁ 3 MODULE TUNING CỦA DỰ ÁN ---")

frame_experiment = pd.read_csv(PROCESSED_DATA_PATH)
features_exp = frame_experiment.drop(columns=["diagnosis"])
target_exp = frame_experiment["diagnosis"].map({1: "M", 0: "B"})

# Create explicit split to measure inference time
X_tr_exp, X_te_exp, y_tr_exp, y_te_exp = train_test_split(
    features_exp, target_exp, test_size=0.2, random_state=42, stratify=target_exp
)

# 1. TUNING CRITERION
print("\n1. Tuning Criterion (Gini vs Entropy)")
t0 = time.perf_counter()
gini_ent_result = run_gini_vs_entropy_experiment(
    features=features_exp, target=target_exp, config=BaselineConfig(test_size=0.2, random_state=42), cv_folds=5
)
t1 = time.perf_counter()
sklearn_family = gini_ent_result.families['sklearn']
best_crit = sklearn_family.selected_criterion
train_acc_crit = sklearn_family.train_metrics[best_crit].accuracy
test_acc_crit = sklearn_family.selected_test_metrics.accuracy

best_model_crit = sklearn_family.runs[best_crit].estimator
t_infer_0 = time.perf_counter()
best_model_crit.predict(X_te_exp)
infer_time_crit = time.perf_counter() - t_infer_0

print(f"  - Tiêu chí tốt nhất: {best_crit}")
print(f"  - Thời gian tuning: {t1 - t0:.4f} giây")
print(f"  - Thời gian suy diễn (Inference Time - Test Set): {infer_time_crit:.6f} giây")
print(f"  - Overfitting Gap (Train Acc - Test Acc): {train_acc_crit - test_acc_crit:.4f}")
print(f"  - Test Accuracy: {test_acc_crit:.4f}")


# 2. TUNING MAX DEPTH
print("\n2. Tuning Max Depth")
t0 = time.perf_counter()
max_depth_cfg = MaxDepthExperimentConfig(depths=(None, 3, 5, 7, 8, 9, 10), test_size=0.2)
max_depth_result = run_max_depth_experiment(
    features=features_exp, target=target_exp, config=max_depth_cfg
)
t1 = time.perf_counter()
best_depth = max_depth_result.selected_depths['sklearn']

best_model_depth = max_depth_result.selected_models['sklearn']
t_infer_0 = time.perf_counter()
best_model_depth.predict(X_te_exp)
infer_time_depth = time.perf_counter() - t_infer_0

best_depth_row = max_depth_result.final_comparison[
    (max_depth_result.final_comparison['implementation'] == 'sklearn') & 
    (max_depth_result.final_comparison['variant'] == 'selected_max_depth')
]
if not best_depth_row.empty:
    train_acc_md = best_depth_row['train_accuracy'].values[0]
    test_acc_md = best_depth_row['test_accuracy'].values[0]
    print(f"  - Độ sâu tốt nhất: {best_depth}")
    print(f"  - Thời gian tuning: {t1 - t0:.4f} giây")
    print(f"  - Thời gian suy diễn (Inference Time - Test Set): {infer_time_depth:.6f} giây")
    print(f"  - Overfitting Gap: {train_acc_md - test_acc_md:.4f}")
    print(f"  - Test Accuracy: {test_acc_md:.4f}")
else:
    print(f"  - Độ sâu tốt nhất: {best_depth} (Thời gian tuning: {t1 - t0:.4f} giây)")


# 3. TUNING MIN SAMPLES
print("\n3. Tuning Min Samples (Split & Leaf)")
t0 = time.perf_counter()
min_samples_cfg = MinSamplesConfig(test_size=0.2, random_state=42)
min_samples_result = run_min_samples_tuning(
    features=features_exp, target=target_exp, config=min_samples_cfg
)
t1 = time.perf_counter()

best_model_ms = min_samples_result.best_model
t_infer_0 = time.perf_counter()
best_model_ms.predict(X_te_exp)
infer_time_ms = time.perf_counter() - t_infer_0

best_ms = min_samples_result.best_candidate
print(f"  - Cấu hình tốt nhất: min_samples_split={best_ms.min_samples_split}, min_samples_leaf={best_ms.min_samples_leaf}")
print(f"  - Thời gian tuning toàn grid: {t1 - t0:.4f} giây")
print(f"  - Thời gian suy diễn (Inference Time - Test Set): {infer_time_ms:.6f} giây")
print(f"  - Overfitting Gap: {best_ms.train_metrics.accuracy - best_ms.test_metrics.accuracy:.4f}")
print(f"  - Test Accuracy: {best_ms.test_metrics.accuracy:.4f}")


--- ĐÁNH GIÁ 3 MODULE TUNING CỦA DỰ ÁN ---

1. Tuning Criterion (Gini vs Entropy)
  - Tiêu chí tốt nhất: entropy
  - Thời gian tuning: 10.1810 giây
  - Thời gian suy diễn (Inference Time - Test Set): 0.000888 giây
  - Overfitting Gap (Train Acc - Test Acc): 0.0702
  - Test Accuracy: 0.9298

2. Tuning Max Depth
  - Độ sâu tốt nhất: 7
  - Thời gian tuning: 27.4370 giây
  - Thời gian suy diễn (Inference Time - Test Set): 0.000988 giây
  - Overfitting Gap: 0.0833
  - Test Accuracy: 0.9123

3. Tuning Min Samples (Split & Leaf)
  - Cấu hình tốt nhất: min_samples_split=5, min_samples_leaf=2
  - Thời gian tuning toàn grid: 1.3963 giây
  - Thời gian suy diễn (Inference Time - Test Set): 0.000978 giây
  - Overfitting Gap: 0.0811
  - Test Accuracy: 0.9123


# 4.2 Đánh giá thời gian và Overfitting của Module Tuning (Raw Data)\nThực hiện lại bài test phần 4.1 trên tập Raw Data.

In [ ]:
import time
from sklearn.model_selection import train_test_split
from app.ml.sklearn_tree.max_depth import run_max_depth_experiment, MaxDepthExperimentConfig
from app.ml.sklearn_tree.gini_vs_entropy import run_gini_vs_entropy_experiment
from app.ml.sklearn_tree.min_samples import run_min_samples_tuning, MinSamplesConfig
from app.ml.sklearn_tree.baseline import BaselineConfig
from app.ml.preprocessing.breast_cancer import load_breast_cancer_dataset

print("--- 4.2 ĐÁNH GIÁ 3 MODULE TUNING (RAW DATA) ---")

ds_raw = load_breast_cancer_dataset(RAW_DATA_PATH)
features_raw = ds_raw.features
target_raw = ds_raw.target

# Create explicit split to measure inference time
X_tr_raw, X_te_raw, y_tr_raw, y_te_raw = train_test_split(
    features_raw, target_raw, test_size=0.2, random_state=42, stratify=target_raw
)

# 1. TUNING CRITERION
print("\n1. Tuning Criterion (Gini vs Entropy)")
t0 = time.perf_counter()
gini_ent_result = run_gini_vs_entropy_experiment(
    features=features_raw, target=target_raw, config=BaselineConfig(test_size=0.2, random_state=42), cv_folds=5
)
t1 = time.perf_counter()
sklearn_family = gini_ent_result.families['sklearn']
best_crit = sklearn_family.selected_criterion
train_acc_crit = sklearn_family.train_metrics[best_crit].accuracy
test_acc_crit = sklearn_family.selected_test_metrics.accuracy

best_model_crit = sklearn_family.runs[best_crit].estimator
t_infer_0 = time.perf_counter()
best_model_crit.predict(X_te_raw)
infer_time_crit = time.perf_counter() - t_infer_0

print(f"  - Tiêu chí tốt nhất: {best_crit}")
print(f"  - Thời gian tuning: {t1 - t0:.4f} giây")
print(f"  - Thời gian suy diễn (Inference Time - Test Set): {infer_time_crit:.6f} giây")
print(f"  - Overfitting Gap (Train Acc - Test Acc): {train_acc_crit - test_acc_crit:.4f}")
print(f"  - Test Accuracy: {test_acc_crit:.4f}")


# 2. TUNING MAX DEPTH
print("\n2. Tuning Max Depth")
t0 = time.perf_counter()
max_depth_cfg = MaxDepthExperimentConfig(depths=(None, 3, 5, 7, 10), test_size=0.2)
max_depth_result = run_max_depth_experiment(
    features=features_raw, target=target_raw, config=max_depth_cfg
)
t1 = time.perf_counter()
best_depth = max_depth_result.selected_depths['sklearn']

best_model_depth = max_depth_result.selected_models['sklearn']
t_infer_0 = time.perf_counter()
best_model_depth.predict(X_te_raw)
infer_time_depth = time.perf_counter() - t_infer_0

best_depth_row = max_depth_result.final_comparison[
    (max_depth_result.final_comparison['implementation'] == 'sklearn') & 
    (max_depth_result.final_comparison['variant'] == 'selected_max_depth')
]
if not best_depth_row.empty:
    train_acc_md = best_depth_row['train_accuracy'].values[0]
    test_acc_md = best_depth_row['test_accuracy'].values[0]
    print(f"  - Độ sâu tốt nhất: {best_depth}")
    print(f"  - Thời gian tuning: {t1 - t0:.4f} giây")
    print(f"  - Thời gian suy diễn (Inference Time - Test Set): {infer_time_depth:.6f} giây")
    print(f"  - Overfitting Gap: {train_acc_md - test_acc_md:.4f}")
    print(f"  - Test Accuracy: {test_acc_md:.4f}")
else:
    print(f"  - Độ sâu tốt nhất: {best_depth} (Thời gian tuning: {t1 - t0:.4f} giây)")


# 3. TUNING MIN SAMPLES
print("\n3. Tuning Min Samples (Split & Leaf)")
t0 = time.perf_counter()
min_samples_cfg = MinSamplesConfig(test_size=0.2, random_state=42)
min_samples_result = run_min_samples_tuning(
    features=features_raw, target=target_raw, config=min_samples_cfg
)
t1 = time.perf_counter()

best_model_ms = min_samples_result.best_model
t_infer_0 = time.perf_counter()
best_model_ms.predict(X_te_raw)
infer_time_ms = time.perf_counter() - t_infer_0

best_ms = min_samples_result.best_candidate
print(f"  - Cấu hình tốt nhất: min_samples_split={best_ms.min_samples_split}, min_samples_leaf={best_ms.min_samples_leaf}")
print(f"  - Thời gian tuning toàn grid: {t1 - t0:.4f} giây")
print(f"  - Thời gian suy diễn (Inference Time - Test Set): {infer_time_ms:.6f} giây")
print(f"  - Overfitting Gap: {best_ms.train_metrics.accuracy - best_ms.test_metrics.accuracy:.4f}")
print(f"  - Test Accuracy: {best_ms.test_metrics.accuracy:.4f}")


# 5. Kiểm thử bộ tham số tối ưu (Optimal Hyperparameters)\nKiểm tra hiệu suất với bộ tham số tốt nhất.

In [37]:
print("--- KIỂM THỬ BỘ THAM SỐ TỐI ƯU NHẤT ---")
optimal_params = {'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 5, 'min_samples_split': 2}
print(f"Tham số: {optimal_params}\n")

X_tr, X_te, y_tr, y_te = train_test_split(X_grid, y_grid, test_size=0.2, random_state=42, stratify=y_grid)

model_opt = DecisionTreeClassifier(random_state=42, **optimal_params)

start_time = time.perf_counter()
model_opt.fit(X_tr, y_tr)
fit_time = time.perf_counter() - start_time

y_tr_pred = model_opt.predict(X_tr)
y_te_pred = model_opt.predict(X_te)

tr_acc = accuracy_score(y_tr, y_tr_pred)
te_acc = accuracy_score(y_te, y_te_pred)
te_f1 = f1_score(y_te, y_te_pred)
te_recall = recall_score(y_te, y_te_pred)
te_prec = precision_score(y_te, y_te_pred)

print(f"Thời gian Fit mô hình: {fit_time:.5f} giây")
print(f"Train Accuracy: {tr_acc:.4f}")
print(f"Test Accuracy:  {te_acc:.4f}")
print(f"Overfitting Gap (Train - Test): {tr_acc - te_acc:.4f}")
print(f"\n--- Metrics trên tập Test ---")
print(f"F1 Score:  {te_f1:.4f}")
print(f"Recall:    {te_recall:.4f}")
print(f"Precision: {te_prec:.4f}")


--- KIỂM THỬ BỘ THAM SỐ TỐI ƯU NHẤT ---
Tham số: {'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 5, 'min_samples_split': 2}

Thời gian Fit mô hình: 0.00843 giây
Train Accuracy: 0.9890
Test Accuracy:  0.9474
Overfitting Gap (Train - Test): 0.0416

--- Metrics trên tập Test ---
F1 Score:  0.9250
Recall:    0.8810
Precision: 0.9737


# 6. Đo lường độc lập Baseline B0\nKiểm tra thời gian huấn luyện và độ quá khớp cho mô hình cơ sở không bị kiểm soát.

In [41]:
print("--- KIỂM THỬ THỜI GIAN CHẠY CỦA BASELINE B0 (KHÔNG GIỚI HẠN) ---")
baseline_params = {'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
print(f"Tham số Baseline: {baseline_params}\n")

# Use X_grid, y_grid from earlier cells
X_tr_b0, X_te_b0, y_tr_b0, y_te_b0 = train_test_split(X_grid, y_grid, test_size=0.2, random_state=42, stratify=y_grid)

model_b0 = DecisionTreeClassifier(random_state=42, **baseline_params)

# Đo thời gian huấn luyện
start_time_b0 = time.perf_counter()
model_b0.fit(X_tr_b0, y_tr_b0)
fit_time_b0 = time.perf_counter() - start_time_b0

# Đo thời gian dự đoán (Inference Latency)
start_infer_b0 = time.perf_counter()
y_te_pred_b0 = model_b0.predict(X_te_b0)
infer_time_b0 = time.perf_counter() - start_infer_b0

y_tr_pred_b0 = model_b0.predict(X_tr_b0)

tr_acc_b0 = accuracy_score(y_tr_b0, y_tr_pred_b0)
te_acc_b0 = accuracy_score(y_te_b0, y_te_pred_b0)

print(f"Thời gian Huấn luyện (Fit Time): {fit_time_b0:.5f} giây")
print(f"Thời gian Suy diễn (Inference Time - trên tập Test): {infer_time_b0:.5f} giây")
print(f"Train Accuracy: {tr_acc_b0:.4f}")
print(f"Test Accuracy:  {te_acc_b0:.4f}")
print(f"Overfitting Gap (Train - Test): {tr_acc_b0 - te_acc_b0:.4f}")
print(f"Số lượng Node Lá (Leaves): {model_b0.get_n_leaves()}")
print(f"Độ sâu của cây (Depth): {model_b0.get_depth()}")


--- KIỂM THỬ THỜI GIAN CHẠY CỦA BASELINE B0 (KHÔNG GIỚI HẠN) ---
Tham số Baseline: {'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}

Thời gian Huấn luyện (Fit Time): 0.00905 giây
Thời gian Suy diễn (Inference Time - trên tập Test): 0.00057 giây
Train Accuracy: 1.0000
Test Accuracy:  0.9035
Overfitting Gap (Train - Test): 0.0965
Số lượng Node Lá (Leaves): 21
Độ sâu của cây (Depth): 8
